# 12 — Train MediaPipe Upper-Body Graph + Spatial/Temporal Transformer

Notebook này đọc packed raw keypoints do notebook 11 tạo. Mỗi epoch sinh một temporal crop và perturbation mới, sau đó ánh xạ đúng thành `[64,68,9]`: **Graph Encoder (2 block) → Spatial Transformer (2 layer) → learned joint pooling → Temporal Transformer (3 layer) → Classifier**.

Training dùng AdamW, warmup + cosine learning rate, label smoothing và early stopping theo validation loss. Checkpoint tốt nhất được khôi phục trước khi đánh giá. Test chỉ được đọc sau khi chọn checkpoint.


In [ ]:
#@title Configuration
PROJECT_GIT_REF = 'feat/kaggle-vsl-mediapipe-graph'  #@param {type:'string'}
CLASS_COUNT = 70  #@param {type:'integer'}
RUN_NAME = 'mediapipe_upper68_pose_transformer_top70_v2'  #@param {type:'string'}
MAX_EPOCHS = 100  #@param {type:'integer'}
BATCH_SIZE = 16  #@param {type:'integer'}
LEARNING_RATE = 0.0005  #@param {type:'number'}
WEIGHT_DECAY = 0.05  #@param {type:'number'}
LABEL_SMOOTHING = 0.10  #@param {type:'number'}
DROPOUT = 0.25  #@param {type:'number'}
WARMUP_EPOCHS = 5  #@param {type:'integer'}
EARLY_STOPPING_PATIENCE = 15  #@param {type:'integer'}
EARLY_STOPPING_MIN_DELTA = 0.0  #@param {type:'number'}
GRAPH_DIM = 128  #@param {type:'integer'}
GRAPH_BLOCKS = 2  #@param {type:'integer'}
SPATIAL_LAYERS = 2  #@param {type:'integer'}
SPATIAL_HEADS = 4  #@param {type:'integer'}
TEMPORAL_DIM = 256  #@param {type:'integer'}
TEMPORAL_LAYERS = 3  #@param {type:'integer'}
TEMPORAL_HEADS = 4  #@param {type:'integer'}
COPY_PACKED_KEYPOINTS_TO_LOCAL = True  #@param {type:'boolean'}
RESUME = True  #@param {type:'boolean'}
RUN_TEST_AFTER_TRAINING = True  #@param {type:'boolean'}
SEED = 42  #@param {type:'integer'}

assert GRAPH_DIM % SPATIAL_HEADS == 0
assert TEMPORAL_DIM % TEMPORAL_HEADS == 0


In [ ]:
#@title Mount Drive and define artifacts
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/silent-signal-results/vsl_kaggle_mediapipe')
SUBSET_ROOT = DRIVE_ROOT / f'subsets/top{CLASS_COUNT}'
MANIFEST = SUBSET_ROOT / 'manifest.csv'
LABELS = SUBSET_ROOT / 'labels.json'
SELECTION = SUBSET_ROOT / 'selection.json'
DRIVE_KEYPOINTS = SUBSET_ROOT / 'keypoints/mediapipe76_front.npz'
PACK_REPORT = SUBSET_ROOT / 'keypoints/mediapipe76_front.report.json'
RUN_ROOT = SUBSET_ROOT / 'models' / RUN_NAME
FIGURES_ROOT = RUN_ROOT / 'figures'
LOCAL_REPO = Path('/content/silent-signal')
LOCAL_KEYPOINTS = Path(f'/content/mediapipe76_front_top{CLASS_COUNT}.npz')

for required in (MANIFEST, LABELS, SELECTION, DRIVE_KEYPOINTS, PACK_REPORT):
    if not required.is_file():
        raise FileNotFoundError(f'Run notebook 11 mới trước; thiếu: {required}')
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
print('Run output:', RUN_ROOT)


In [ ]:
#@title Checkout code and install training dependencies
import subprocess, sys
if not (LOCAL_REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch',
        'https://github.com/stillthethrone/silent-signal.git', str(LOCAL_REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'fetch', 'origin', PROJECT_GIT_REF], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'checkout', '-B', PROJECT_GIT_REF, f'origin/{PROJECT_GIT_REF}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{LOCAL_REPO}[training]'], check=True)
PROJECT_COMMIT = subprocess.check_output(
    ['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
print('Commit:', PROJECT_COMMIT)


In [ ]:
#@title Verify GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before training.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
#@title Copy one packed NPZ from Drive to local disk
import hashlib, json, shutil, time

def _sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

pack_report = json.loads(PACK_REPORT.read_text(encoding='utf-8'))
MANIFEST_SHA256 = _sha256(MANIFEST)
if pack_report.get('status') != 'complete':
    raise RuntimeError(f'Pack report chưa hoàn chỉnh: {pack_report}')
if pack_report.get('manifest_sha256') != MANIFEST_SHA256:
    raise RuntimeError('Packed keypoints không khớp manifest hiện tại; chạy lại notebook 11.')
expected_keypoints_sha256 = pack_report.get('output_sha256')
if not expected_keypoints_sha256 or _sha256(DRIVE_KEYPOINTS) != expected_keypoints_sha256:
    raise RuntimeError('Packed keypoints trên Drive không khớp SHA-256 trong report.')

if COPY_PACKED_KEYPOINTS_TO_LOCAL:
    started = time.perf_counter()
    partial = LOCAL_KEYPOINTS.with_suffix(LOCAL_KEYPOINTS.suffix + '.partial')
    local_is_current = (
        LOCAL_KEYPOINTS.is_file()
        and LOCAL_KEYPOINTS.stat().st_size == DRIVE_KEYPOINTS.stat().st_size
        and _sha256(LOCAL_KEYPOINTS) == expected_keypoints_sha256
    )
    if not local_is_current:
        if partial.exists():
            partial.unlink()
        shutil.copyfile(DRIVE_KEYPOINTS, partial)
        if partial.stat().st_size != DRIVE_KEYPOINTS.stat().st_size:
            raise RuntimeError('Packed keypoint copy is incomplete.')
        if _sha256(partial) != expected_keypoints_sha256:
            partial.unlink()
            raise RuntimeError('Packed keypoint copy failed SHA-256 verification.')
        partial.replace(LOCAL_KEYPOINTS)
    KEYPOINTS = LOCAL_KEYPOINTS
    print(f'Local copy ready in {(time.perf_counter() - started) / 60:.1f} min')
else:
    KEYPOINTS = DRIVE_KEYPOINTS
print('Training keypoints:', KEYPOINTS)
print('Size GiB:', round(KEYPOINTS.stat().st_size / 1024**3, 3))


In [ ]:
#@title Validate the 76-point pack and the new 68×9 representation
import json
from silent_signal.data.manifest import read_manifest
from silent_signal.data.keypoint_pack import read_packed_keypoints
from silent_signal.preprocessing.mediapipe_features import (
    MediaPipeFeatureConfig,
    mediapipe_graph_features,
)

packed = read_packed_keypoints(KEYPOINTS)
manifest_sample_ids = tuple(sorted(record.sample_id for record in read_manifest(MANIFEST)))
packed_sample_ids = tuple(sorted(packed.sample_ids))
if packed.metadata.get('manifest_sha256') != MANIFEST_SHA256:
    raise RuntimeError('Metadata trong NPZ không khớp manifest hiện tại.')
if packed_sample_ids != manifest_sample_ids:
    missing = sorted(set(manifest_sample_ids) - set(packed_sample_ids))
    extra = sorted(set(packed_sample_ids) - set(manifest_sample_ids))
    raise RuntimeError(
        f'Packed sample IDs không khớp chính xác manifest: '
        f'missing={missing[:3]}, extra={extra[:3]}'
    )
if pack_report.get('samples') != len(packed.sample_ids):
    raise RuntimeError('Sample count trong pack report không khớp NPZ.')
feature_config = MediaPipeFeatureConfig(target_frames=64)
sample_features, sample_joint_mask, sample_frame_mask = mediapipe_graph_features(
    packed.sequence(0), feature_config
)
assert sample_features.shape == (64, 68, 9), sample_features.shape
assert len(packed.sample_ids) > 0
print('Packed clips:', len(packed.sample_ids))
print('Sample:', packed.sample_ids[0], sample_features.shape)
print('Pack metadata:', json.dumps(packed.metadata, ensure_ascii=False, indent=2))


In [ ]:
#@title Train with warmup/cosine, resume, and early stopping
import subprocess, sys
command = [
    sys.executable, '-u', '-m', 'silent_signal.cli.train_pose_transformer',
    '--manifest', str(MANIFEST),
    '--keypoints', str(KEYPOINTS),
    '--output-root', str(RUN_ROOT),
    '--project-commit', PROJECT_COMMIT,
    '--view', 'front',
    '--target-frames', '64',
    '--epochs', str(MAX_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--weight-decay', str(WEIGHT_DECAY),
    '--label-smoothing', str(LABEL_SMOOTHING),
    '--warmup-epochs', str(WARMUP_EPOCHS),
    '--patience', str(EARLY_STOPPING_PATIENCE),
    '--min-delta', str(EARLY_STOPPING_MIN_DELTA),
    '--seed', str(SEED),
    '--workers', '2',
    '--device', 'cuda',
    '--graph-dim', str(GRAPH_DIM),
    '--graph-blocks', str(GRAPH_BLOCKS),
    '--spatial-layers', str(SPATIAL_LAYERS),
    '--spatial-heads', str(SPATIAL_HEADS),
    '--temporal-dim', str(TEMPORAL_DIM),
    '--temporal-layers', str(TEMPORAL_LAYERS),
    '--temporal-heads', str(TEMPORAL_HEADS),
    '--feedforward-ratio', '2',
    '--dropout', str(DROPOUT),
]
if not RESUME:
    command.append('--no-resume')
if RUN_TEST_AFTER_TRAINING:
    command.append('--run-test')
print('+', ' '.join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
#@title Reports and comparison plots
import csv, json
import matplotlib.pyplot as plt
import numpy as np

report = json.loads((RUN_ROOT / 'report.json').read_text(encoding='utf-8'))
history = json.loads((RUN_ROOT / 'history.json').read_text(encoding='utf-8'))
names = report['data']['class_names']
print(json.dumps({
    key: report[key]
    for key in (
        'parameters', 'epochs_completed', 'best_epoch',
        'best_validation_loss', 'stopped_early', 'evaluation',
    )
}, ensure_ascii=False, indent=2))

epochs = [row['epoch'] for row in history]
best_epoch = report['best_epoch']
figure, axes = plt.subplots(2, 3, figsize=(18, 9))
axes[0, 0].plot(epochs, [row['train_loss'] for row in history], label='train')
axes[0, 0].plot(epochs, [row['validation_loss'] for row in history], label='validation')
axes[0, 0].set(title='Loss (cross-entropy)', xlabel='epoch')
axes[0, 1].plot(epochs, [row['train_top1'] for row in history], label='train')
axes[0, 1].plot(epochs, [row['validation_top1'] for row in history], label='validation')
axes[0, 1].set(title='Top-1', xlabel='epoch', ylim=(0, 1))
axes[0, 2].plot(epochs, [row['validation_top5'] for row in history], label='validation')
axes[0, 2].set(title='Top-5', xlabel='epoch', ylim=(0, 1))
axes[1, 0].plot(epochs, [row['validation_macro_f1'] for row in history], label='validation')
axes[1, 0].set(title='Macro-F1', xlabel='epoch', ylim=(0, 1))
axes[1, 1].plot(
    epochs,
    [row['train_top1'] - row['validation_top1'] for row in history],
    label='train − validation Top-1',
)
axes[1, 1].axhline(0, color='gray', linewidth=1)
axes[1, 1].set(title='Generalization gap', xlabel='epoch')
axes[1, 2].plot(epochs, [row['learning_rate'] for row in history], color='purple')
axes[1, 2].set(title='Learning rate', xlabel='epoch')
for axis in axes.flat:
    axis.axvline(best_epoch, color='gray', linestyle=':', label='best')
    axis.grid(alpha=0.25)
    handles, labels = axis.get_legend_handles_labels()
    if handles:
        axis.legend(fontsize=8)
figure.suptitle(
    f'Pose Transformer top-{CLASS_COUNT} — best epoch {best_epoch}, '
    f'val loss {report["best_validation_loss"]:.3f}'
)
figure.tight_layout()
figure.savefig(FIGURES_ROOT / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

available_splits = [
    split for split in ('validation', 'test') if split in report.get('evaluation', {})
]
for split in available_splits:
    confusion_path = RUN_ROOT / f'confusion_{split}.csv'
    per_class_path = RUN_ROOT / f'per_class_{split}.csv'
    missing = [path for path in (confusion_path, per_class_path) if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Report có split {split} nhưng thiếu artifact: {missing}')
    confusion = np.loadtxt(
        confusion_path, delimiter=',', dtype=np.int64
    )
    normalized = confusion / np.maximum(confusion.sum(axis=1, keepdims=True), 1)
    figure, axis = plt.subplots(figsize=(18, 16))
    image = axis.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
    axis.set_xticks(range(len(names)), names, rotation=90, fontsize=7)
    axis.set_yticks(range(len(names)), names, fontsize=7)
    axis.set(
        title=f'Normalized confusion matrix — {split}',
        xlabel='Predicted', ylabel='True',
    )
    figure.colorbar(image, ax=axis, fraction=0.03)
    figure.tight_layout()
    figure.savefig(FIGURES_ROOT / f'confusion_{split}.png', dpi=160)
    plt.show()

    with per_class_path.open(encoding='utf-8') as handle:
        per_class = sorted(csv.DictReader(handle), key=lambda row: float(row['f1']))
    macro_f1 = report['evaluation'][split]['macro_f1']
    colors = [
        '#d62728' if float(row['f1']) < 0.4
        else '#ffb000' if float(row['f1']) < macro_f1
        else '#2ca02c'
        for row in per_class
    ]
    figure, axis = plt.subplots(figsize=(10, max(10, len(per_class) * 0.28)))
    axis.barh(
        [f"{row['gloss']} (n={row['support']})" for row in per_class],
        [float(row['f1']) for row in per_class],
        color=colors,
    )
    axis.axvline(
        macro_f1, color='blue', linestyle='--', label=f'Macro-F1={macro_f1:.3f}'
    )
    axis.set(title=f'F1-score by class — {split}', xlabel='F1-score', xlim=(0, 1))
    axis.tick_params(axis='y', labelsize=7)
    axis.grid(axis='x', alpha=0.25)
    axis.legend()
    figure.tight_layout()
    figure.savefig(FIGURES_ROOT / f'per_class_f1_{split}.png', dpi=160)
    plt.show()

    pairs = [
        (confusion[i, j], names[i], names[j])
        for i in range(len(names))
        for j in range(len(names))
        if i != j and confusion[i, j] > 0
    ]
    print(f'\nTop confusion pairs ({split}): true → predicted')
    for count, truth, predicted in sorted(pairs, reverse=True)[:15]:
        print(f'  {truth} → {predicted}: {count}')

print('Figures saved to:', FIGURES_ROOT)


## Reading overfitting

- `MAX_EPOCHS=100` chỉ là giới hạn trên; early stopping có thể dừng sớm.
- Checkpoint cuối cùng dùng để báo cáo là `best_checkpoint.pt`, được chọn hoàn toàn bằng validation loss.
- Overfit bắt đầu khi train loss tiếp tục giảm nhưng validation loss tăng, đồng thời khoảng cách Top‑1 mở rộng.
- Khi đổi kiến trúc hoặc hyperparameter, hãy đổi `RUN_NAME`. Resume cùng thư mục chỉ dành cho đúng một cấu hình.
- Nếu T4 báo hết VRAM, đổi `BATCH_SIZE` từ 16 xuống 8; không thay các kích thước model trước.
